In [7]:
import time
import json
import random
import sqlite3
import socket
import paho.mqtt.client as mqtt
from datetime import datetime

# ── Settings ──────────────────────────────────────────────────
BROKER_IP    = "172.20.10.2"
BROKER_PORT  = 1883
TOPIC        = "Message"
DB_PATH      = r"D:\Cyber_Project\database\ids.db"
rate_per_sec = 5
duration_sec = 60

# ════════════════════════════════════════════════════════════
#  STEP 1: Get real local IP
# ════════════════════════════════════════════════════════════
def get_local_ip():
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        ip = s.getsockname()[0]
        s.close()
        return ip
    except:
        return "127.0.0.1"

LOCAL_IP       = get_local_ip()
FIXED_SRC_PORT = random.randint(49152, 65535)  # fixed for entire session

print(f"📡 Detected local IP : {LOCAL_IP}")
print(f"📡 Fixed source port : {FIXED_SRC_PORT}")

# ════════════════════════════════════════════════════════════
#  STEP 2: Check broker reachability BEFORE connecting
# ════════════════════════════════════════════════════════════
def check_broker_reachable(ip, port, timeout=5):
    print(f"🔍 Checking broker {ip}:{port} ...", end=" ")
    try:
        sock = socket.create_connection((ip, port), timeout=timeout)
        sock.close()
        print("✅ Reachable!")
        return True
    except socket.timeout:
        print("❌ TIMED OUT — broker not responding")
    except ConnectionRefusedError:
        print("❌ CONNECTION REFUSED — broker offline or wrong port")
    except OSError as e:
        print(f"❌ NETWORK ERROR — {e}")
    return False

# ════════════════════════════════════════════════════════════
#  STEP 3: MQTT callbacks for diagnostics
# ════════════════════════════════════════════════════════════
def on_connect(client, userdata, flags, rc):
    codes = {
        0: "✅ Connected successfully",
        1: "❌ Wrong protocol version",
        2: "❌ Invalid client ID",
        3: "❌ Broker unavailable",
        4: "❌ Bad credentials",
        5: "❌ Not authorised",
    }
    print(codes.get(rc, f"❌ Unknown error code: {rc}"))

def on_disconnect(client, userdata, rc):
    if rc != 0:
        print(f"⚠️  Unexpected disconnect (rc={rc}). Retrying...")

# ════════════════════════════════════════════════════════════
#  STEP 4: Dashboard DB write
# ════════════════════════════════════════════════════════════
def show_on_dashboard(payload):
    try:
        conn = sqlite3.connect(DB_PATH)
        conn.execute("""
            INSERT INTO alerts
            (timestamp, src_ip, dst_ip,
             src_port, dst_port, protocol,
             attack_type, confidence, source)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            payload.get("src_ip",      LOCAL_IP),
            payload.get("dst_ip",      BROKER_IP),
            payload.get("src_port",    FIXED_SRC_PORT),
            payload.get("dst_port",    BROKER_PORT),
            payload.get("protocol",    "MQTT"),
            payload.get("attack_type", "BENIGN"),
            payload.get("confidence",  0.90),
            "Traffic_Generator"
        ))
        conn.commit()
        conn.close()
    except Exception as e:
        print(f"⚠️  DB write error: {e}")

# ════════════════════════════════════════════════════════════
#  MAIN
# ════════════════════════════════════════════════════════════
if not check_broker_reachable(BROKER_IP, BROKER_PORT):
    print("\n🛠️  Troubleshooting tips:")
    print(f"   1. Is your broker device (IP: {BROKER_IP}) ON and on the same network?")
    print(f"   2. Run on broker machine:  netstat -an | findstr 1883")
    print(f"   3. Check firewall — port 1883 must be open on broker")
    print(f"   4. Ping test — run: ping {BROKER_IP}")
    print(f"   5. If using hotspot, confirm IP hasn't changed")
    raise SystemExit("❌ Aborting — broker unreachable.")

# ── MQTT client setup ─────────────────────────────────────────
client = mqtt.Client(client_id="pc-traffic-generator")
client.on_connect    = on_connect
client.on_disconnect = on_disconnect

try:
    client.connect(BROKER_IP, BROKER_PORT, keepalive=60)
except Exception as e:
    raise SystemExit(f"❌ MQTT connect failed: {e}")

client.loop_start()
time.sleep(1)  # allow connection to establish

# ── Traffic loop ──────────────────────────────────────────────
start = time.time()
count = 0

print(f"\n🚀 Sending traffic → Dashboard: http://localhost:5000")
print(f"   Broker  : {BROKER_IP}:{BROKER_PORT}")
print(f"   Src IP  : {LOCAL_IP}")
print(f"   Src Port: {FIXED_SRC_PORT}")
print(f"   Rate    : {rate_per_sec}/sec  Duration: {duration_sec}s\n")

while time.time() - start < duration_sec:

    payload = {
        # ── Original fields ──────────────────────────────────
        "src"   : "PC",
        "count" : count,
        "value" : random.randint(1, 100),
        "ts"    : time.time(),

        # ── Corrected network fields ─────────────────────────
        "src_ip"      : LOCAL_IP,           # ✅ real local IP
        "dst_ip"      : BROKER_IP,          # ✅ broker IP
        "src_port"    : FIXED_SRC_PORT,     # ✅ stable port (no random)
        "dst_port"    : BROKER_PORT,        # ✅ MQTT port 1883
        "protocol"    : "MQTT",
        "attack_type" : "BENIGN",
        "confidence"  : round(random.uniform(0.88, 0.97), 4),
    }

    # ── Publish to broker ────────────────────────────────────
    result = client.publish(TOPIC, json.dumps(payload))
    if result.rc != mqtt.MQTT_ERR_SUCCESS:
        print(f"⚠️  Publish failed (rc={result.rc}) — broker may have dropped")

    # ── Write to dashboard DB ────────────────────────────────
    show_on_dashboard(payload)

    # ── Progress log every 5 messages ───────────────────────
    count += 1
    if count % 5 == 0:
        print(f"  [{time.time()-start:5.1f}s] "
              f"Msg #{count:3} | "
              f"src: {LOCAL_IP}:{FIXED_SRC_PORT} → "
              f"{BROKER_IP}:{BROKER_PORT} | "
              f"value: {payload['value']}")

    time.sleep(1.0 / rate_per_sec)

# ── Cleanup ───────────────────────────────────────────────────
client.loop_stop()
client.disconnect()
print(f"\n✅ Done. Published {count} messages.")
print(f"   View at: http://localhost:5000/alerts")

📡 Detected local IP : 172.20.10.2
📡 Fixed source port : 65137
🔍 Checking broker 172.20.10.2:1883 ... ✅ Reachable!
✅ Connected successfully


C:\Users\johna\AppData\Local\Temp\ipykernel_16036\2235079187.py:113: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id="pc-traffic-generator")



🚀 Sending traffic → Dashboard: http://localhost:5000
   Broker  : 172.20.10.2:1883
   Src IP  : 172.20.10.2
   Src Port: 65137
   Rate    : 5/sec  Duration: 60s

  [  1.4s] Msg #  5 | src: 172.20.10.2:65137 → 172.20.10.2:1883 | value: 75
  [  2.9s] Msg # 10 | src: 172.20.10.2:65137 → 172.20.10.2:1883 | value: 72
  [  4.3s] Msg # 15 | src: 172.20.10.2:65137 → 172.20.10.2:1883 | value: 61
  [  6.0s] Msg # 20 | src: 172.20.10.2:65137 → 172.20.10.2:1883 | value: 3


KeyboardInterrupt: 

⚠️  Unexpected disconnect (rc=7). Retrying...
✅ Connected successfully
⚠️  Unexpected disconnect (rc=7). Retrying...
✅ Connected successfully
⚠️  Unexpected disconnect (rc=7). Retrying...


C:\Users\johna\AppData\Local\Temp\ipykernel_36080\1283582693.py:16: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id="pc-traffic-generator")


🚀 Sending traffic → Dashboard: http://localhost:5000
   Broker : 172.20.10.2:1883
   Rate   : 5/sec  Duration: 60s

  [  1.7s] Msg #  5 | src_ip: PC_Traffic_Generator | dst: 172.20.10.2:1883 | value: 9
  [  3.4s] Msg # 10 | src_ip: PC_Traffic_Generator | dst: 172.20.10.2:1883 | value: 18
  [  5.3s] Msg # 15 | src_ip: PC_Traffic_Generator | dst: 172.20.10.2:1883 | value: 96
  [  6.9s] Msg # 20 | src_ip: PC_Traffic_Generator | dst: 172.20.10.2:1883 | value: 78
  [  8.5s] Msg # 25 | src_ip: PC_Traffic_Generator | dst: 172.20.10.2:1883 | value: 69
  [ 10.2s] Msg # 30 | src_ip: PC_Traffic_Generator | dst: 172.20.10.2:1883 | value: 39
  [ 11.8s] Msg # 35 | src_ip: PC_Traffic_Generator | dst: 172.20.10.2:1883 | value: 42
  [ 13.4s] Msg # 40 | src_ip: PC_Traffic_Generator | dst: 172.20.10.2:1883 | value: 39
  [ 15.0s] Msg # 45 | src_ip: PC_Traffic_Generator | dst: 172.20.10.2:1883 | value: 90
  [ 16.8s] Msg # 50 | src_ip: PC_Traffic_Generator | dst: 172.20.10.2:1883 | value: 96
  [ 18.5s] Msg 

KeyboardInterrupt: 